# 07 Data Integrity

Check numerical outliers, consistency between columns, and employee identifiers.

Identify potential issues without modifying the original data.

## 1. Preparation

In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/processed/06_date_validation.parquet")
df_clean = df.copy()

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 95,000
Columns: 35


## 2. Outlier Detection

Use the IQR method to identify unusual numerical values.

Salary anomalies were already flagged in the previous notebook.

In [2]:
numeric_columns = [
    "age",
    "years_experience",
    "tenure_years",
    "performance_review_score"
]

results = []

for column in numeric_columns:
    values = df[column].dropna()

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = (values < lower) | (values > upper)

    results.append({
        "column": column,
        "lower_bound": round(lower, 2),
        "upper_bound": round(upper, 2),
        "outliers": outliers.sum(),
        "percentage": round(outliers.mean() * 100, 2)
    })

display(pd.DataFrame(results))

,column,lower_bound,upper_bound,outliers,percentage
0,age,9.5,61.5,178,0.19
1,years_experience,-11.5,40.5,0,0.00
2,tenure_years,-5.0,11.0,149,0.16
3,performance_review_score,1.0,5.8,0,0.00


The IQR analysis identified 178 age outliers and 149 tenure outliers.

No outliers were found in experience or performance scores.

All values were preserved since statistical outliers are not necessarily errors.

## 3. Cross-Column Consistency

Check for logical conflicts between employee age, experience, and tenure.

In [3]:
consistency_checks = {
    "Experience exceeds working age": (
        df_clean["years_experience"] > df_clean["age"] - 16
    ),
    "Tenure exceeds experience": (
        df_clean["tenure_years"] > df_clean["years_experience"]
    ),
    "Tenure exceeds working age": (
        df_clean["tenure_years"] > df_clean["age"] - 16
    )
}

results = pd.DataFrame({
    "check": consistency_checks.keys(),
    "flagged_records": [
        mask.fillna(False).sum()
        for mask in consistency_checks.values()
    ]
})

display(results)

,check,flagged_records
0,Experience exceeds working age,0
1,Tenure exceeds experience,0
2,Tenure exceeds working age,0


No logical inconsistencies were found between age, experience, and tenure across 95,000 employee records.

## 4. Unique Identifier Validation

Check employee IDs for duplicates, missing values, and invalid formats.

In [4]:
ids = df_clean["employee_id"].astype("string")

checks = {
    "Total records": len(ids),
    "Unique IDs": ids.nunique(),
    "Duplicate IDs": ids.duplicated().sum(),
    "Missing IDs": ids.isna().sum(),
    "Empty IDs": ids.str.strip().eq("").sum(),
    "Invalid formats": (
        ~ids.str.fullmatch(r"(E\d+|LEG-\d+)").fillna(False)
    ).sum()
}

display(
    pd.DataFrame(
        checks.items(),
        columns=["check", "count"]
    )
)

invalid_ids = ~ids.str.fullmatch(r"(E\d+|LEG-\d+)").fillna(False)

if invalid_ids.any():
    display(df_clean.loc[invalid_ids, ["employee_id"]].head(20))

,check,count
0,Total records,95000
1,Unique IDs,95000
2,Duplicate IDs,0
3,Missing IDs,0
4,Empty IDs,0
5,Invalid formats,0


## 5. Export

All integrity checks are complete. No data changes were needed.

In [5]:
from pathlib import Path

assert len(df_clean) == 95000
assert df_clean["employee_id"].is_unique
assert df_clean["employee_id"].notna().all()

pd.testing.assert_frame_equal(df, df_clean)

output_path = Path("data/processed/07_data_integrity.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_parquet(output_path, index=False)

saved = pd.read_parquet(output_path)
pd.testing.assert_frame_equal(df_clean, saved)

print(f"Saved: {output_path}")
print(f"Rows: {len(saved):,}")
print(f"Columns: {len(saved.columns)}")

Saved: data/processed/07_data_integrity.parquet
Rows: 95,000
Columns: 35
